In [28]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: Bautista TUPM-26-3506
Seed Digit: 6
ID Sum: 22
Name Length: 8
Random Seed: 263506
Verification Value: 182


In [29]:
#[CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measuremnent_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measuremnent_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:" , WORK_DIR.resolve())
print("All datasets files verified.")

Workspace: C:\Users\jacob\Documents\Comprog\Lab7_NumPy_Pandas_Bautista\Lab7_NumPy_Pandas_Additional_Exercises\mobile_device_usage_profiling
All datasets files verified.


In [30]:
#[CELL 3] NUMPY ARRAYS AND DATASET INSPECTION

demo = np.array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Mulitplication:", demo * 3)

raw_measurements = np.genfromtxt(measuremnent_path, dtype=float)
assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)

print("Missing Values:", np.isnan(raw_measurements).sum())

Sample Array: [2 4 6 8]
Element-wise Mulitplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [31]:
# [CELL 4] - LABLED DATAFRAME
n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

df = pd.DataFrame(
    raw_measurements.copy(), columns=feature_names
)

# Load and attach the labels to create Result and Outcome
labels = np.loadtxt(label_path, usecols=0, dtype=int)
assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

# Now create profile_df with Record_ID, features, Result, and Outcome included
profile_df = df.copy()
profile_df.insert(
    0,
    "Record_ID",
    profile_df.index
)

print("Shape:", profile_df.shape)
print(profile_df.head())


Shape: (1567, 593)
   Record_ID  Feature_000  Feature_001  Feature_002  Feature_003  Feature_004  \
0          0      3030.93      2564.00    2187.7333    1411.1265       1.3602   
1          1      3095.78      2465.14    2230.4222    1463.6606       0.8294   
2          2      2932.61      2559.94    2186.4111    1698.0172       1.5102   
3          3      2988.72      2479.90    2199.0333     909.7926       1.3204   
4          4      3032.24      2502.87    2233.3667    1326.5200       1.5334   

   Feature_005  Feature_006  Feature_007  Feature_008  ...  Feature_582  \
0        100.0      97.6133       0.1242       1.5005  ...       0.5005   
1        100.0     102.3433       0.1247       1.4966  ...       0.5019   
2        100.0      95.4878       0.1241       1.4436  ...       0.4958   
3        100.0     104.2367       0.1217       1.4882  ...       0.4990   
4        100.0     100.3967       0.1235       1.5031  ...       0.4800   

   Feature_583  Feature_584  Feature_585  F

In [32]:
# [CELL 5] - BALANCED SAMPLE

pass_count = (
    profile_df["Outcome"] == "Pass"
).sum()

fail_count = (
    profile_df["Outcome"] == "Fail"
).sum()

balanced_size = min(
    20 + ID_SUM,
    pass_count,
    fail_count
)

balanced_sample = (
    profile_df
    .groupby(
        "Outcome",
        group_keys=False
    )
    .sample(
        n=balanced_size,
        random_state=RANDOM_SEED
    )
)

print("Pass Count:", pass_count)
print("Fail Count:", fail_count)
print("Balanced Size:", balanced_size)

print("\nOutcome Counts")
print(
    balanced_sample["Outcome"]
    .value_counts()
)

Pass Count: 1463
Fail Count: 104
Balanced Size: 42

Outcome Counts
Outcome
Fail    42
Pass    42
Name: count, dtype: int64


In [33]:
# [CELL 6] - SELECTED FEATURES

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

rng = np.random.default_rng(RANDOM_SEED)

selected_positions = rng.choice(
    len(usable_indices),
    size=5,
    replace=False
)

selected_indices = usable_indices[
    selected_positions
]

selected_features = [
    feature_names[i]
    for i in selected_indices
]

analysis_df = balanced_sample[
    ["Record_ID", "Outcome"]
    + selected_features
].copy()

for feature in selected_features:

    analysis_df[feature] = (
        analysis_df[feature]
        .fillna(
            analysis_df[feature].median()
        )
    )

print("Selected Features:")
print(selected_features)

print("\nRemaining Missing Values:")
print(
    analysis_df[selected_features]
    .isna()
    .sum()
    .sum()
)

Selected Features:
['Feature_390', 'Feature_492', 'Feature_581', 'Feature_108', 'Feature_580']

Remaining Missing Values:
0


In [34]:
# [CELL 7] - GROUPED STATISTICS

grouped_stats = (
    analysis_df
    .groupby("Outcome")[selected_features]
    .agg([
        "count",
        "mean",
        "median",
        "std"
    ])
)

print(grouped_stats.round(4))

        Feature_390                         Feature_492                  \
              count    mean  median     std       count    mean  median   
Outcome                                                                   
Fail             42  0.9187  0.8620  0.2793          42  8.3051  8.2812   
Pass             42  0.8150  0.7505  0.2218          42  8.1608  8.2812   

                Feature_581                            Feature_108          \
            std       count     mean   median      std       count    mean   
Outcome                                                                      
Fail     0.4146          42  72.5964  71.8357  24.5415          42 -0.0005   
Pass     0.5627          42  75.0583  71.8357  23.4966          42  0.0053   

                        Feature_580                         
         median     std       count   mean  median     std  
Outcome                                                     
Fail    -0.0109  0.0901          42  0.005  0.0047

In [35]:
# [CELL 8] - MEAN DIFFERENCE AND STANDARDIZED MEAN DIFFERENCE

comparison_rows = []

for feature in selected_features:

    pass_values = analysis_df.loc[
        analysis_df["Outcome"] == "Pass",
        feature
    ]

    fail_values = analysis_df.loc[
        analysis_df["Outcome"] == "Fail",
        feature
    ]

    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    mean_difference = (
        fail_mean - pass_mean
    )

    pooled_sd = np.sqrt(
        (
            (len(pass_values) - 1)
            * pass_values.std(ddof=1)**2
            +
            (len(fail_values) - 1)
            * fail_values.std(ddof=1)**2
        )
        /
        (
            len(pass_values)
            + len(fail_values)
            - 2
        )
    )

    absolute_smd = (
        abs(mean_difference)
        / pooled_sd
        if pooled_sd > 0
        else 0
    )

    comparison_rows.append([
        feature,
        pass_mean,
        fail_mean,
        mean_difference,
        absolute_smd
    ])

In [36]:
# [Cell 9] - FEATURE COMPARISON DATAFRAME

feature_comparison = pd.DataFrame(
    comparison_rows,
    columns=[
        "Feature",
        "Pass_Mean",
        "Fail_Mean",
        "Mean_Difference",
        "Absolute_SMD"
    ]
)

feature_comparison = (
    feature_comparison
    .sort_values(
        by="Absolute_SMD",
        ascending=False
    )
    .reset_index(drop=True)
)

comparison_path = (
    OUTPUT_DIR /
    "feature_comparison.csv"
)

feature_comparison.to_csv(
    comparison_path,
    index=False
)

print(feature_comparison.round(4))

       Feature  Pass_Mean  Fail_Mean  Mean_Difference  Absolute_SMD
0  Feature_390     0.8150     0.9187           0.1037        0.4112
1  Feature_492     8.1608     8.3051           0.1443        0.2920
2  Feature_581    75.0583    72.5964          -2.4618        0.1025
3  Feature_108     0.0053    -0.0005          -0.0058        0.0693
4  Feature_580     0.0050     0.0050          -0.0000        0.0151


In [37]:
# [Cell 10] - VERIFY EXPORT

loaded_comparison = pd.read_csv(
    comparison_path
)

pd.testing.assert_frame_equal(
    loaded_comparison,
    feature_comparison,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10
)

highest_feature = (
    feature_comparison.iloc[0]["Feature"]
)

highest_smd = (
    feature_comparison.iloc[0]["Absolute_SMD"]
)

print("Balanced Sample Size:")
print(balanced_size)

print("\nSelected Features:")
print(selected_features)

print("\nHighest-Ranked Feature:")
print(highest_feature)

print("\nAbsolute Standardized Mean Difference:")
print(round(highest_smd, 4))

print("\nNumPy-Pandas Statistics Match:")
print(True)

print("\nExported File:")
print(comparison_path.name)

Balanced Sample Size:
42

Selected Features:
['Feature_390', 'Feature_492', 'Feature_581', 'Feature_108', 'Feature_580']

Highest-Ranked Feature:
Feature_390

Absolute Standardized Mean Difference:
0.4112

NumPy-Pandas Statistics Match:
True

Exported File:
feature_comparison.csv
